# El 311 no cabe en una columna

**Nivel:** beginner

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jcval94/narrative/blob/codex/corporate-data-narrative-lab/corporate-data-narrative-lab/outputs/notebooks/12-el-311-no-cabe-en-una-columna.ipynb)

## Pregunta central

Que solicitudes conviene revisar antes de automatizar el ruteo de atencion ciudadana?

## Recreación narrativa

En la junta alguien quiere comprar un clasificador para mandar cada reporte 311 al area correcta. Nadia pide abrir primero la tabla: si falta el descriptor, se rompe la ubicacion o un canal domina la captura, el modelo aprenderia a decir depende con presupuesto.

*La escena es una recreación; las conclusiones provienen del dataset citado.*

## Fuente real

**311 Service Requests from 2010 to Present, sample Jan 1-7 2024**, NYC Open Data. [Página de origen](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9) · [datos](https://data.cityofnewyork.us/resource/erm2-nwe9.csv?%24select=unique_key%2Ccreated_date%2Cclosed_date%2Cagency%2Cagency_name%2Ccomplaint_type%2Cdescriptor%2Cstatus%2Cincident_zip%2Cincident_address%2Cborough%2Clatitude%2Clongitude%2Cresolution_description%2Copen_data_channel_type&%24where=created_date+between+%272024-01-01T00%3A00%3A00%27+and+%272024-01-08T00%3A00%3A00%27&%24order=unique_key&%24limit=5000) · licencia: NYC Open Data Terms of Use.  
Consultado: 2026-07-18 · 5,000 filas · columnas usadas: `unique_key`, `created_date`, `closed_date`, `agency`, `agency_name`, `complaint_type`, `descriptor`, `status`, `incident_zip`, `incident_address`, `borough`, `latitude`, `longitude`, `resolution_description`, `open_data_channel_type`.

In [1]:
# @title Preparar los datos { display-mode: "form" }
borough_filter = "Todos" # @param ["Todos", "BROOKLYN", "QUEENS", "MANHATTAN", "BRONX", "STATEN ISLAND"]
DATA_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.csv?%24select=unique_key%2Ccreated_date%2Cclosed_date%2Cagency%2Cagency_name%2Ccomplaint_type%2Cdescriptor%2Cstatus%2Cincident_zip%2Cincident_address%2Cborough%2Clatitude%2Clongitude%2Cresolution_description%2Copen_data_channel_type&%24where=created_date+between+%272024-01-01T00%3A00%3A00%27+and+%272024-01-08T00%3A00%3A00%27&%24order=unique_key&%24limit=5000"
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import Markdown, display
df_completo = pd.read_csv(DATA_URL)
df_completo["created_date"] = pd.to_datetime(df_completo["created_date"], errors="coerce")
df_completo["closed_date"] = pd.to_datetime(df_completo["closed_date"], errors="coerce")
df_completo["dias_abierto"] = (df_completo["closed_date"] - df_completo["created_date"]).dt.total_seconds() / 86400
estado_orden = {"Open": 1, "Assigned": 2, "In Progress": 3, "Closed": 4}
df_completo["status_rank"] = df_completo["status"].map(estado_orden).astype("Int64")
df = df_completo if borough_filter == "Todos" else df_completo[df_completo["borough"] == borough_filter]
df = df.reset_index(drop=True)
display(Markdown(f"**Filtro global:** {borough_filter} - **{len(df):,} solicitudes**"))
df.head()

**Filtro global:** Todos - **5,000 solicitudes**

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,status,incident_zip,incident_address,borough,latitude,longitude,resolution_description,open_data_channel_type,dias_abierto,status_rank
0,59886871,2024-01-01 00:04:14,2024-01-01 01:00:13,NYPD,New York City Police Department,Illegal Parking,Blocked Hydrant,Closed,10468.0,2535 GRAND AVENUE,BRONX,40.865311,-73.901382,The Police Department issued a summons in resp...,MOBILE,0.038877,4
1,59886906,2024-01-01 00:59:23,2024-01-01 01:27:12,NYPD,New York City Police Department,Illegal Parking,Posted Parking Sign Violation,Closed,11373.0,81-09 41 AVENUE,QUEENS,40.745916,-73.884193,The Police Department responded to the complai...,PHONE,0.019317,4
2,59886911,2024-01-01 00:52:54,2024-01-01 01:09:19,NYPD,New York City Police Department,Illegal Parking,Posted Parking Sign Violation,Closed,11224.0,2928 MARSHA RAPAPORT WAY,BROOKLYN,40.578226,-73.972346,The Police Department responded to the complai...,MOBILE,0.011400,4
3,59887007,2024-01-01 00:44:31,2024-01-01 01:03:11,NYPD,New York City Police Department,Blocked Driveway,No Access,Closed,11436.0,144-28 LINDEN BOULEVARD,QUEENS,40.684755,-73.799314,The Police Department responded to the complai...,PHONE,0.012963,4
4,59887011,2024-01-01 00:41:16,2024-01-01 01:17:24,NYPD,New York City Police Department,Blocked Driveway,Partial Access,Closed,11239.0,598 SCHROEDERS AVENUE,BROOKLYN,40.655993,-73.870408,The Police Department responded to the complai...,ONLINE,0.025093,4


## 1. Tipos de variables

La primera trampa es tratar todo como si fuera numero. Una fecha no se promedia como espera, y un texto no se agrupa sin decidir que significa.

**Pregunta:** Que cambia cuando una columna es numerica, categorica, ordinal, fecha o texto?

**Conexión:** Punto de partida

In [2]:
tipo_variables = pd.DataFrame([
    ["dias_abierto", "numerica", df["dias_abierto"].dropna().round(2).head(1).to_string(index=False)],
    ["complaint_type", "categorica", df["complaint_type"].dropna().iloc[0]],
    ["status_rank", "ordinal", df["status_rank"].dropna().head(1).to_string(index=False)],
    ["created_date", "fecha", df["created_date"].dropna().dt.date.astype(str).iloc[0]],
    ["resolution_description", "texto", df["resolution_description"].dropna().str.slice(0, 55).iloc[0]],
], columns=["variable", "tipo", "ejemplo"])
tipo_variables

,variable,tipo,ejemplo
0,dias_abierto,numerica,0.04
1,complaint_type,categorica,Illegal Parking
2,status_rank,ordinal,4
3,created_date,fecha,2024-01-01
4,resolution_description,texto,The Police Department issued a summons in resp...


In [3]:
# @title Explorar Tipos de variables { display-mode: "form" }
variables_meta = pd.DataFrame({
    "variable": ["dias_abierto", "complaint_type", "status_rank", "created_date", "resolution_description"],
    "tipo": ["numerica", "categorica", "ordinal", "fecha", "texto"],
    "completos": [df[c].notna().sum() for c in ["dias_abierto", "complaint_type", "status_rank", "created_date", "resolution_description"]],
    "faltantes_pct": [round(df[c].isna().mean() * 100, 2) for c in ["dias_abierto", "complaint_type", "status_rank", "created_date", "resolution_description"]],
})
fig = px.bar(variables_meta, x="variable", y="completos", hover_data=["tipo", "faltantes_pct"], title="Tipos de variable y cobertura")
fig.update_layout(updatemenus=[{"buttons": [
    {"label": "registros completos", "method": "update", "args": [{"y": [variables_meta["completos"]]}, {"yaxis": {"title": "registros completos"}}]},
    {"label": "% faltante", "method": "update", "args": [{"y": [variables_meta["faltantes_pct"]]}, {"yaxis": {"title": "% faltante"}}]},
]}])
fig.show()


display(Markdown("**Lo que muestra:** La misma tabla mezcla numero, categoria, orden, fecha y texto; cada tipo pide operaciones distintas."))

**Lo que muestra:** La misma tabla mezcla numero, categoria, orden, fecha y texto; cada tipo pide operaciones distintas.

## 2. Calidad de datos

El proveedor pregunta por el algoritmo. Nadia pregunta por lo aburrido: faltantes, duplicados, rangos y canales. La junta sufre, pero la tabla respira.

**Pregunta:** Que problemas aparecen antes de creerle a la tabla?

**Conexión:** Tipos de variables mostro que cada columna se interpreta distinto; ahora revisamos si esas columnas son confiables.

In [4]:
lat = pd.to_numeric(df["latitude"], errors="coerce")
lon = pd.to_numeric(df["longitude"], errors="coerce")
zip_num = pd.to_numeric(df["incident_zip"], errors="coerce")
huella = ["created_date", "complaint_type", "descriptor", "incident_address", "borough"]
canal = df["open_data_channel_type"].value_counts()
casos = [df["descriptor"].isna().sum(), df.duplicated(huella).sum(), (lat.isna() | lon.isna() | ~lat.between(40.45, 40.95) | ~lon.between(-74.30, -73.65) | (zip_num.notna() & ~zip_num.between(10000, 11699))).sum(), canal.max()]
calidad = pd.DataFrame({"revision": ["faltantes", "duplicados", "rangos_invalidos", f"sesgo_medicion_{canal.idxmax()}"], "casos": casos})
calidad.assign(pct=(calidad["casos"] / len(df) * 100).round(2))

,revision,casos,pct
0,faltantes,266,5.32
1,duplicados,27,0.54
2,rangos_invalidos,44,0.88
3,sesgo_medicion_ONLINE,2166,43.32


In [5]:
# @title Explorar Calidad de datos { display-mode: "form" }
calidad_plot = calidad.assign(pct=(calidad["casos"] / len(df) * 100).round(2)).sort_values("casos")
fig = px.bar(calidad_plot, x="casos", y="revision", orientation="h", hover_data=["pct"], title="Chequeos basicos de calidad")
fig.update_layout(updatemenus=[{"buttons": [
    {"label": "casos", "method": "update", "args": [{"x": [calidad_plot["casos"]]}, {"xaxis": {"title": "casos"}}]},
    {"label": "porcentaje", "method": "update", "args": [{"x": [calidad_plot["pct"]]}, {"xaxis": {"title": "% de solicitudes"}}]},
]}])
fig.show()


display(Markdown("**Lo que muestra:** Los problemas no son teoricos: hay faltantes, huellas repetidas, ubicaciones que no pasan reglas simples y un canal dominante."))

**Lo que muestra:** Los problemas no son teoricos: hay faltantes, huellas repetidas, ubicaciones que no pasan reglas simples y un canal dominante.

## 3. Preparacion basica

Operacion no necesita una conferencia: necesita una lista corta de que revisar primero. Data filtra lo imposible, transforma texto, agrupa y ordena.

**Pregunta:** Como usamos filtrar, ordenar, agrupar y transformar sin esconder el problema?

**Conexión:** Calidad de datos dejo claro que no basta mirar la tabla cruda; ahora limpiamos lo minimo para decidir.

In [6]:
preparacion = df.dropna(subset=["complaint_type"]).query("dias_abierto >= 0")
preparacion = preparacion.assign(descriptor_limpio=preparacion["descriptor"].fillna("SIN DESCRIPTOR").str.upper())
preparacion = preparacion.groupby(["borough", "complaint_type"], as_index=False).agg(casos=("unique_key", "count"), mediana_dias=("dias_abierto", "median"))
preparacion = preparacion.sort_values(["casos", "mediana_dias"], ascending=[False, False])
preparacion.head(10)

,borough,complaint_type,casos,mediana_dias
91,BROOKLYN,Illegal Parking,350,0.042546
244,QUEENS,Noise - Residential,311,0.029942
104,BROOKLYN,Noise - Residential,300,0.021950
234,QUEENS,Illegal Parking,293,0.125787
30,BRONX,Noise - Residential,217,0.034097
19,BRONX,HEAT/HOT WATER,207,0.899109
179,MANHATTAN,Noise - Residential,192,0.019242
209,QUEENS,Blocked Driveway,179,0.123426
59,BROOKLYN,Blocked Driveway,147,0.045081
22,BRONX,Illegal Parking,132,0.109699


In [7]:
# @title Explorar Preparacion basica { display-mode: "form" }
top_prep = preparacion.head(15).assign(etiqueta=lambda t: t["borough"] + " | " + t["complaint_type"])
fig = px.bar(top_prep.sort_values("casos"), x="casos", y="etiqueta", orientation="h", hover_data=["mediana_dias"], title="Categorias priorizadas despues de preparar")
fig.update_layout(updatemenus=[{"buttons": [
    {"label": "volumen", "method": "update", "args": [{"x": [top_prep["casos"]]}, {"xaxis": {"title": "solicitudes"}}]},
    {"label": "mediana dias", "method": "update", "args": [{"x": [top_prep["mediana_dias"]]}, {"xaxis": {"title": "mediana de dias abiertos"}}]},
]}])
fig.show()


display(Markdown("**Lo que muestra:** La preparacion no maquilla: convierte una tabla grande en una cola de decisiones trazables."))

**Lo que muestra:** La preparacion no maquilla: convierte una tabla grande en una cola de decisiones trazables.

## Cómo se conecta todo

Primero distinguimos que tipo de cosa contiene cada columna. Luego revisamos si esa columna se puede creer. Al final preparamos una tabla pequena para decidir que categorias revisar con Operacion antes de automatizar.

## Decisión

No comprar el clasificador todavia: revisar descriptores faltantes, huellas duplicadas, ubicaciones invalidas y canales de captura antes de priorizar categorias.

**Regla:** Antes de transformar datos, pregunta que mide cada variable y que parte de la realidad se quedo fuera.